# Pipeline de Preproducción — Pharma Sales Forecast

Este notebook contiene el pipeline de transformación de datos y entrenamiento final optimizado para producción a escala semanal.

In [ ]:
import os
import re
import json
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from dateutil.relativedelta import relativedelta

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

In [ ]:
csv_path = PROJECT_ROOT / "02_datos" / "01_Originales" / "salesweekly.csv"
df = pd.read_csv(csv_path)

def normalize_column_name(col: str) -> str:
    col = str(col).strip().lower()
    col = unicodedata.normalize("NFKD", col)
    col = "".join(ch for ch in col if not unicodedata.combining(ch))
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col

df.columns = [normalize_column_name(c) for c in df.columns]

df['date'] = pd.to_datetime(df['datum'], errors='coerce')
df = df.sort_values('date').reset_index(drop=True)

max_date = df['date'].max()
cutoff_date = max_date - relativedelta(months=3)
train = df[df['date'] < cutoff_date].copy()
validation = df[df['date'] >= cutoff_date].copy()

In [ ]:
df_train = train.copy()
date_series = pd.to_datetime(df_train['date'])
df_train['year'] = date_series.dt.year
df_train['month'] = date_series.dt.month
df_train['day'] = date_series.dt.day
df_train['weekofyear'] = date_series.dt.isocalendar().week.astype(int)

target_cols = ['m01ab', 'm01ae', 'n02ba', 'n02be', 'n05b', 'n05c', 'r03', 'r06']

for target in target_cols:
    lag_1 = df_train[target].shift(1)
    lag_2 = df_train[target].shift(2)
    roll_4 = lag_1.rolling(window=4, min_periods=1).mean()
    
    df_train[f'{target}_lag_1'] = lag_1
    df_train[f'{target}_lag_2'] = lag_2
    df_train[f'{target}_roll_mean_4'] = roll_4

In [ ]:
with open(str(PROJECT_ROOT / '06_resultados' / 'Modelizacion' / 'config_mejor_modelo.json'), 'r', encoding='utf-8') as f:
    best_config = json.load(f)['mejor_configuracion_por_target']

trained_models = {}
for target in target_cols:
    target_config = best_config[target]
    algoritmo = target_config['algoritmo']
    params = target_config['parametros']
    
    features_target = ['year', 'month', 'day', 'weekofyear', f'{target}_lag_1', f'{target}_lag_2', f'{target}_roll_mean_4']
    X_target = df_train[features_target]
    y_target = df_train[target]
    
    non_nan_mask = X_target.notna().all(axis=1)
    X_target = X_target[non_nan_mask].reset_index(drop=True)
    y_target = y_target[non_nan_mask].reset_index(drop=True)
    
    if algoritmo == 'HistGradientBoostingRegressor':
        model = HistGradientBoostingRegressor(random_state=42, **params)
    elif algoritmo == 'RandomForestRegressor':
        model = RandomForestRegressor(random_state=42, **params)
        
    model.fit(X_target, y_target)
    trained_models[target] = model
    print(f"Modelo final {algoritmo} entrenado para {target} con {len(X_target)} filas.")